# SkyGuard AI - Supervised Machine Learning Training
This notebook trains a highly accurate Supervised **Random Forest Classifier** to detect hardware faults (Spikes, Drifts, Freezes, Physics Violations) across 7 diverse Indian climate zones. 
**Features:** Spatial Awareness (One-Hot Encoding) and Temporal Differencing (Lag features).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_curve, auc
import joblib
import os

print("Libraries loaded successfully.")

## 1. Data Loading & Feature Engineering

In [ ]:
# Update path if running on Kaggle!
DATA_PATH = '/kaggle/input/skyguard-national-benchmark/national_benchmark_injected.csv'
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'national_benchmark_injected.csv'
    
print(f"Loading dataset from {DATA_PATH}...")
df = pd.read_csv(DATA_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])

# --- FEATURE ENGINEERING ---
# 1. Time Features
hour = df['timestamp'].dt.hour
df['hour_sin'] = np.sin(2 * np.pi * hour / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * hour / 24.0)

# 2. Spatial Features (One-Hot Encoding for Climate Awareness)
df = pd.get_dummies(df, columns=['station'], prefix='loc')
loc_cols = [c for c in df.columns if c.startswith('loc_')]
for col in loc_cols:
    df[col] = df[col].astype(int)

# 3. Temporal Differencing (Crucial for Random Forest to detect 'Spikes' and 'Freezes')
df['temp_diff'] = df['temperature_c'].diff().fillna(0)
df['rh_diff'] = df['relative_humidity_pct'].diff().fillna(0)
df['pres_diff'] = df['surface_pressure_hpa'].diff().fillna(0)

# 4. Target Variable (0 = Normal, 1 = Anomaly)
df['true_anomaly'] = (df['anomaly_label'] > 0).astype(int)

print(f"Total records loaded: {len(df)}")
print("\nTarget Distribution:")
print(df['true_anomaly'].value_counts())

## 2. Train Random Forest Model

In [ ]:
features = ['temperature_c', 'relative_humidity_pct', 'surface_pressure_hpa', 'dew_point_c', 
            'hour_sin', 'hour_cos', 'temp_diff', 'rh_diff', 'pres_diff'] + loc_cols

X = df[features].values
y = df['true_anomaly'].values

# Split into Train and Test sets (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1, class_weight='balanced')
rf_model.fit(X_train, y_train)
print("Training Complete!")

# Predict on the Test Set
y_pred = rf_model.predict(X_test)
y_scores = rf_model.predict_proba(X_test)[:, 1]

# Also save predictions for the full dataset for the time-series plot
df['rf_pred'] = rf_model.predict(X)

## 3. Evaluation & Visualization

In [ ]:
print("\n--- Random Forest Model Evaluation (Test Set) ---")
print(classification_report(y_test, y_pred))
f1 = f1_score(y_test, y_pred)
print(f"Final Test F1-Score: {f1:.4f}\n")

# 1. Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal', 'Anomaly'], 
            yticklabels=['Normal', 'Anomaly'])
plt.title('SkyGuard ML - Confusion Matrix (Random Forest)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# 2. Plot ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_scores)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

# 3. Feature Importances
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 6))
plt.title("Feature Importances (What the AI looks at)")
plt.bar(range(X.shape[1]), importances[indices], align="center")
plt.xticks(range(X.shape[1]), [features[i] for i in indices], rotation=45, ha='right')
plt.xlim([-1, X.shape[1]])
plt.tight_layout()
plt.show()

# 4. Time Series Anomaly Plot (Full Dataset Slice)
anomaly_indices = df[df['true_anomaly'] == 1].index
if len(anomaly_indices) > 0:
    start_idx = max(0, anomaly_indices[0] - 200)
    end_idx = min(len(df), start_idx + 1000)
    plot_df = df.iloc[start_idx:end_idx].reset_index(drop=True)
    
    plt.figure(figsize=(15, 6))
    plt.plot(plot_df.index, plot_df['temperature_c'], label='Temperature (°C)', color='blue', alpha=0.6)
    
    anomalies_pred = plot_df[plot_df['rf_pred'] == 1]
    plt.scatter(anomalies_pred.index, anomalies_pred['temperature_c'], color='red', s=40, label='RF Detected Anomaly', zorder=5)
    
    plt.title('Time-Series Anomaly Detection Visualization (Random Forest)')
    plt.xlabel('Time Step (Hours)')
    plt.ylabel('Temperature (°C)')
    plt.legend()
    plt.show()

## 4. Export Model for SkyGuard UI

In [ ]:
out_dir = '/kaggle/working/skyguard_models/'
os.makedirs(out_dir, exist_ok=True)

joblib.dump(rf_model, os.path.join(out_dir, 'rf_model.pkl'))

# Save the exact feature list so the app.py dashboard knows exactly what columns to pass to the model
import json
with open(os.path.join(out_dir, 'rf_features.json'), 'w') as f:
    json.dump({'features': features}, f)

print(f"Model and features list successfully exported to {out_dir}")
print("Download these files and place them in the 'models/saved_weights/' directory of your local repository.")